In [1]:
%pip install -U langchain langchain-core langchain-community langchain-openai langchain-openrouter langchain-qdrant langchain-text-splitters qdrant-client python-dotenv pypdf pymupdf

Note: you may need to restart the kernel to use updated packages.


# Loading Documents

In [2]:
from langchain_core.documents import Document

doc = Document(
    page_content=(
        "Full-time employees receive 24 days of paid annual leave per calendar year. "
        "Up to 5 unused days may be carried over into the next year."
    ),
    metadata={"filename": "employee_handbook.pdf", "page": 1, "area": "policies"},
)

print(doc.page_content)
print(doc.metadata)

Full-time employees receive 24 days of paid annual leave per calendar year. Up to 5 unused days may be carried over into the next year.
{'filename': 'employee_handbook.pdf', 'page': 1, 'area': 'policies'}


## Loading text and Markdown

In [ ]:
from langchain_community.document_loaders import TextLoader, UnstructuredMarkdownLoader

notes = TextLoader("knowledge/wiki/onboarding.md", encoding="utf-8").load()

print(len(notes))
print(notes[0].metadata)
print(notes[0].page_content)




Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
1
{'source': 'knowledge/wiki/onboarding.md'}
# Engineering Onboarding

Welcome to Zomato Engineering. This page covers your first weeks. Ask in #new-joiners if anything is unclear.

## Day one

- Collect your laptop and access card from the IT desk.
- Meet your onboarding buddy. They will walk you through the team's codebase and rituals.
- Complete the mandatory security awareness course in the Learning Portal within your first 3 days.
- Join the team Slack channels listed in your welcome email.

## Accounts and access

You will receive these accounts on day one: company email, Slack, GitHub, Jira, Confluence and the internal Zomato Admin dashboard. Production access is **not** granted automatically. Request it through the Access Portal after you finish the security course; your manager must approve it, and access is reviewed every quarter.

### VPN setup


In [ ]:
%pip install unstructured
%pip install markdown

In [14]:
notes_md = UnstructuredMarkdownLoader("knowledge/wiki/onboarding.md", mode="elements").load()

print(len(notes_md))
print(notes_md[0])

44
page_content='Engineering Onboarding' metadata={'source': 'knowledge/wiki/onboarding.md', 'category_depth': 0, 'languages': ['eng'], 'file_directory': 'knowledge/wiki', 'filename': 'onboarding.md', 'filetype': 'text/markdown', 'last_modified': '2026-10-08T20:29:36', 'category': 'Title', 'element_id': '5538e4f54984a9265fafa7d6d6b4943b'}


## Loading a PDF

In [22]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("knowledge/policies/employee_handbook.pdf").load()

print(f"total pages: {len(pages)}")

for i, p in enumerate(pages):
    print(f"page: {i+1} \n{p}")

total pages: 8
page: 1 
page_content='Zomato Employee Handbook
Version 4.2 · Effective 1 April 2026 · Owner: People Operations
1. Welcome to Zomato
Zomato connects millions of customers with restaurants and delivery partners every day. We are a technology company at
heart, and our work touches food, logistics, payments and support. This handbook explains how we work, what you can
expect from us and what we expect from you. If anything here conflicts with your employment contract, the contract
applies.
2. Our values
Customer obsession. Start with what the customer, restaurant partner or delivery partner needs, then work
backwards.
Ownership. If you see a problem, you own it until it is fixed or handed over clearly.
Speed with care. We move fast, and we do not cut corners on food safety, payments or privacy.
Simple and honest. Say what you mean, share bad news early and prefer plain language over jargon.
Respect for everyone. Every person who works with Zomato, from a rider to a director

In [24]:
%pip install pdfplumber

  Using cached pdfplumber-0.11.10-py3-none-any.whl.metadata (43 kB)
  Using cached pdfminer_six-20260107-py3-none-any.whl.metadata (4.3 kB)
  Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl.metadata (4.4 kB)
  Using cached cffi-2.1.1-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
Using cached pdfplumber-0.11.10-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20260107-py3-none-any.whl (6.6 MB)
Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl (3.8 MB)
Using cached cffi-2.1.1-cp312-cp312-win_amd64.whl (185 kB)
Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl (7.2 MB)

   ------ --------------------------------- 1/6 [Pillow]
   ------ --------------------------------- 1/6 [Pillow]
   ------ --------------------------------- 1/6 [Pillow]
   ------ --------------------------------- 1/6 [Pillow]
   ------ --------------------------------- 1/6 [Pillow]
   ------ --------------------------------- 1/6

In [26]:
import time
from langchain_community.document_loaders import PDFPlumberLoader, PyMuPDFLoader, PyPDFLoader

path = "knowledge/policies/expense_policy.pdf"      # a PDF that contains a table
candidates = {"pypdf": PyPDFLoader, "pymupdf": PyMuPDFLoader, "pdfplumber": PDFPlumberLoader}

for name, loader_cls in candidates.items():
    started = time.perf_counter()
    pages = loader_cls(path).load()
    seconds = time.perf_counter() - started
    chars = sum(len(p.page_content) for p in pages)
    print(f"{name:>10}: {len(pages)} pages, {chars:,} characters, {seconds:.2f}s")
    print("            sample:", pages[2].page_content[:90].replace("\n", " | "))

     pypdf: 5 pages, 6,239 characters, 0.05s
            sample: 6. Domestic travel | Item Rule | Flights Economy class. Book at least 7 days ahead where possi
   pymupdf: 5 pages, 6,239 characters, 0.01s
            sample: 6. Domestic travel | Item | Rule | Flights | Economy class. Book at least 7 days ahead where possi
pdfplumber: 5 pages, 6,244 characters, 0.18s
            sample: 6. Domestic travel | Item Rule | Flights Economy class. Book at least 7 days ahead where possi


### Scanned PDFs need OCR

In [ ]:
%pip install "unstructured[pdf]"
# OCR also needs system tools: tesseract and poppler (for example: brew install tesseract poppler)

  Using cached googleapis_common_protos-1.75.5-py3-none-any.whl.metadata (8.5 kB)
  Using cached opentelemetry_api-1.45.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.7.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.66.1-cp312-cp312-win_amd64.whl.metadata (132 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.

In [4]:
from langchain_community.document_loaders import UnstructuredPDFLoader
import os
import shutil

os.environ["PATH"] = (
    r"C:\Program Files\Tesseract-OCR"
    + os.pathsep
    + os.environ.get("PATH", "")
)
print(shutil.which("tesseract"))  # should show the path to tesseract.exe

scan = UnstructuredPDFLoader(
    "knowledge/scans/vendor_contract.pdf",
    # mode="elements",            # titles, paragraphs and tables as separate Documents
    strategy="ocr_only",        # force OCR instead of reading a text layer
).load()

print(len(scan), "elements")
print(scan[0].metadata.get("category"), "|", scan[0].page_content)

No languages specified, defaulting to English.


C:\Program Files\Tesseract-OCR\tesseract.EXE
1 elements
None | MASTER SERVICES AGREEMENT

between ZOMATO LIMITED (“Zomato”) and FRESHROUTE LOGISTICS PRIVATE LIMITED (“Vendor”)

This Master Services Agreement (the “Agreement”) is made on 12 January 2026 at Gurugram between Zomato Limited, having its registered office in Gurugram, Haryana, and FreshRoute Logistics Private Limited, having its registered office in Bengaluru, Karnataka. The Agreement governs the supply of last- mile cold-chain delivery services by the Vendor to Zomato,

1. Definitions

“Services” means the temperature-controlled delivery services described in Clause 3 and Schedule A. “Order” means a single delivery request placed by Zomato. “Promised Window” means the delivery time shown to the customer at checkout. “Monthly Fee” means all fees invoiced for Services delivered in a calendar month, “Confidential Information” has the meaning given in Clause 9.

2. Term

This Agreement starts on | February 2026 and continues fo

## CSV, JSON, Word and web loaders

### CSV: one row, one Document

In [28]:
from langchain_community.document_loaders import CSVLoader

faq = CSVLoader(
    "knowledge/faq/support_faq.csv",
    encoding="utf-8",
    metadata_columns=["product", "category"],
).load()

print(len(faq), "rows")
print(faq[0].page_content)
print(faq[0].metadata)

63 rows
question: How do I get a refund for a cancelled order?
answer: Refunds for cancelled orders go back to the original payment method within 5 to 7 working days. Zomato credits are refunded instantly.
{'source': 'knowledge/faq/support_faq.csv', 'row': 0, 'product': 'Zomato App', 'category': 'Refunds'}


### JSON: choose what becomes text

In [2]:
%pip install jq

Note: you may need to restart the kernel to use updated packages.


In [5]:

from langchain_community.document_loaders import JSONLoader

# ".plans[]" -> one Document per plan (3 plans = 3 Documents)
plans = JSONLoader(
    file_path="knowledge/catalog/plans.json",
    jq_schema=".plans[]",
    text_content=False,          # keep each plan as JSON text
).load()

print(len(plans), "documents")   # 3
print(plans[0].page_content)     # the Starter plan, as JSON

# ".plans[] | {name, sso}" -> each Document keeps only those two fields
compact = JSONLoader(
    file_path="knowledge/catalog/plans.json",
    jq_schema=".plans[] | {name, sso}",
    text_content=False,
).load()

# ".billing" -> a single Document holding just the billing block
billing = JSONLoader(
    file_path="knowledge/catalog/plans.json",
    jq_schema=".billing",
    text_content=False,
).load()

# metadata_func: lift a field into metadata so you can filter on it later
def plan_meta(record: dict, metadata: dict) -> dict:
    metadata["plan"] = record.get("name")
    return metadata

per_plan = JSONLoader(
    file_path="knowledge/catalog/plans.json",
    jq_schema=".plans[]",
    text_content=False,
    metadata_func=plan_meta,
).load()
print(per_plan[1].metadata["plan"])   # Business

3 documents
{"name": "Starter", "price_per_employee_per_month": 99, "min_employees": 10, "sso": false, "scim": false, "support": "Email support, response within 2 business days", "features": ["Company meal wallet", "Daily spending limits", "Monthly consolidated invoice", "Email support"]}
Business


## Word, HTML and live web pages

In [7]:
%pip install docx2txt

Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_community.document_loaders import BSHTMLLoader, Docx2txtLoader, WebBaseLoader

memo = Docx2txtLoader("knowledge/policies/travel_memo.docx").load()
page = BSHTMLLoader("knowledge/wiki/holiday_calendar.html").load()
site = WebBaseLoader("https://example.com/help/shipping-rates").load()

print(memo)
print(page)
print(site)

[Document(metadata={'source': 'knowledge/policies/travel_memo.docx'}, page_content='Memo: Changes to business travel booking from 1 May 2026\n\nFrom: Finance and Admin | To: All employees | Date: 15 April 2026\n\nWhy we are making this change\n\nTravel spend has grown 40% over the last year, and many bookings were made directly with airlines and hotels at higher prices. A single booking channel lets us negotiate corporate fares and gives employees support if plans change.\n\nWhat changes\n\nEffective 1 May 2026, all domestic flights and hotels must be booked through the Travel Desk portal instead of directly with airlines or hotels. Train tickets can still be booked on the railway site and claimed as usual.\n\nBookings made outside the portal will not be reimbursed unless the Travel Desk was unavailable and Finance approved the exception in writing.\n\nBooking rules\n\nFlights remain economy class. Book at least 7 days ahead to get the corporate fare. Hotels must stay within the nightl

## Loading a whole folder

In [10]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

loader = DirectoryLoader(
    "knowledge/policies",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True,
)
docs = loader.load()

100%|██████████| 2/2 [00:00<00:00,  7.95it/s]


In [15]:
from ai_eng.settings import KNOWLEDGE_DIR
from ai_eng.loaders import load_corpus

docs = load_corpus(KNOWLEDGE_DIR)

No languages specified, defaulting to English.


Loaded 134 documents from D:\coding shuttle\ai engineer\practice-lab\ai-eng\src\ai_eng\4_langchain_rag\knowledge
